# 3.4 — ROC-AUC

ROC-AUC evaluates your model across ALL possible thresholds at once — giving a threshold-independent measure of quality.

**ROC** = Receiver Operating Characteristic
**AUC** = Area Under the Curve

### Key ideas:
- Model outputs **probabilities** (0.91, 0.08...) via `predict_proba()` — not just 0/1
- A **threshold** (default 0.5) converts those probabilities into hard YES/NO
- F1/precision/recall depend on the threshold — change threshold, numbers change
- AUC measures quality independent of any threshold

### Plain English meaning of AUC:
> Pick one random survivor and one random non-survivor. AUC = the probability the model gave the survivor a higher score.

| AUC | Meaning |
|-----|--------|
| 0.5 | Random guessing — model learned nothing |
| 0.7 | Acceptable |
| 0.8 | Good |
| 0.9 | Excellent |
| 1.0 | Perfect (likely overfitting) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, roc_curve

# Load and prepare Titanic
df = sns.load_dataset('titanic')
cols = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
df = df[cols].copy()
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
df['family_size'] = df['sibsp'] + df['parch'] + 1
df['is_alone'] = (df['family_size'] == 1).astype(int)
df['sex'] = LabelEncoder().fit_transform(df['sex'])
df = pd.get_dummies(df, columns=['embarked'], drop_first=True)

X = df.drop(columns=['survived'])
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model = LogisticRegression(random_state=42)
model.fit(X_train_s, y_train)

# y_pred = hard 0/1 labels — for F1, precision, recall
y_pred = model.predict(X_test_s)

# y_prob = probability scores — for AUC
y_prob = model.predict_proba(X_test_s)[:, 1]

print("Setup complete.")
print("\ny_pred (hard labels):", y_pred[:8])
print("y_prob (probabilities):", y_prob[:8].round(3))

## What predict_proba() Returns

Two columns: [probability of class 0, probability of class 1]. Always sum to 1.

In [ ]:
proba_full = model.predict_proba(X_test_s)
print("Full predict_proba output (first 5 rows):")
print(proba_full[:5].round(3))
print("\nColumn 0 = prob of NOT survived")
print("Column 1 = prob of survived  ← this is y_prob")
print("\nVerify they sum to 1.0:")
print(proba_full[:5].sum(axis=1))

## Understanding AUC — Manual Calculation

AUC = fraction of (survivor, non-survivor) pairs where model gave survivor a higher score.

In [ ]:
# Manual AUC calculation on 4 passengers
passengers = pd.DataFrame({
    'name':     ['Alice', 'Bob', 'Carol', 'Dave'],
    'score':    [0.90,    0.20,   0.70,   0.60],
    'survived': [1,       0,      1,      0]
})

print(passengers)
print()

survivors     = passengers[passengers['survived'] == 1]
non_survivors = passengers[passengers['survived'] == 0]

correct = 0
total   = 0

for _, s in survivors.iterrows():
    for _, ns in non_survivors.iterrows():
        total += 1
        result = s['score'] > ns['score']
        correct += result
        symbol = '✓' if result else '✗'
        print(f"{s['name']} ({s['score']}) vs {ns['name']} ({ns['score']}) → {symbol}")

print(f"\nCorrect: {correct}/{total}")
print(f"Manual AUC = {correct/total:.2f}")

## AUC on Titanic Model

In [ ]:
auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC: {auc:.3f}")
print(f"\nPlain English: If you pick one random survivor and one random")
print(f"non-survivor, the model ranks the survivor higher {auc:.1%} of the time.")

## Plotting the ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {auc:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random baseline (AUC = 0.50)')
plt.fill_between(fpr, tpr, alpha=0.08)
plt.xlabel('False Positive Rate (FPR) → false alarms')
plt.ylabel('True Positive Rate (TPR) → survivors caught')
plt.title('ROC Curve — Titanic Survival')
plt.legend()
plt.tight_layout()
plt.show()

print("The further the curve bows toward top-left → better the model")
print("Curve hugging the diagonal → random guessing")

## AUC vs F1 — Same Model, Different Thresholds

In [ ]:
from sklearn.metrics import f1_score

print(f"AUC = {auc:.3f}  ← stays the same regardless of threshold")
print()
print(f"{'Threshold':>10}  {'F1':>8}  Note")
print("-" * 45)
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_t = (y_prob >= t).astype(int)
    f1 = f1_score(y_test, y_t, zero_division=0)
    note = " ← default" if t == 0.5 else ""
    print(f"{t:>10.1f}  {f1:>8.3f}{note}")

## Comparing Two Models Using AUC

In [ ]:
# Train a second model
model2 = DecisionTreeClassifier(random_state=42)
model2.fit(X_train_s, y_train)
y_prob2 = model2.predict_proba(X_test_s)[:, 1]
auc2 = roc_auc_score(y_test, y_prob2)

# Plot both ROC curves
fpr2, tpr2, _ = roc_curve(y_test, y_prob2)

plt.figure(figsize=(7, 5))
plt.plot(fpr,  tpr,  label=f'Logistic Regression (AUC={auc:.3f})',  linewidth=2)
plt.plot(fpr2, tpr2, label=f'Decision Tree       (AUC={auc2:.3f})', linewidth=2, linestyle='--')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Model Comparison')
plt.legend()
plt.tight_layout()
plt.show()

winner = 'Logistic Regression' if auc > auc2 else 'Decision Tree'
print(f"Logistic Regression AUC: {auc:.3f}")
print(f"Decision Tree AUC:       {auc2:.3f}")
print(f"Better model: {winner}")

## Key Rules

| Situation | Use |
|-----------|-----|
| Comparing two models | AUC |
| Reporting final performance | F1 + classification_report |
| Threshold not yet decided | AUC |
| Need interpretable single metric | F1 |

> **Always use `predict_proba()[:,1]` for AUC — never `predict()`. Hard labels give wrong AUC.**

## Practice Task

In [ ]:
# Q1 — Calculate and print the AUC score for the Titanic model
# YOUR CODE HERE

# Q2 — Plot the ROC curve with the random baseline
# YOUR CODE HERE

# Q3 — Complete this sentence:
# "If I pick one random survivor and one random non-survivor,
#  the model ranks the survivor higher ___% of the time."
# ANSWER:

# Q4 — Train a DecisionTreeClassifier. Which has higher AUC?
# YOUR CODE HERE

# Q5 — Why does AUC stay the same when you change the threshold?
# ANSWER: